# ALBEF ITC contrastive-margin Grad-CAM visualization

Validates and visualizes heatmaps produced by `extract_itc_margin_gradcam_heatmaps.py`. The explained target is `(positive_similarity - negative_similarity) / temperature`.

In [ ]:
from pathlib import Path
import math, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display

pd.set_option('display.max_columns', 100)
TARGET_LABELS = ['Cardiomegaly', 'Pleural effusion']
EXPECTED_METHOD = 'albef_itc_margin_final_self_attention_gradcam'
EXPECTED_VERSION = '2.1-margin-attn-drop-hook'
EXPECTED_IMAGE_RES = 256
NUM_IMAGES = 50
CASES_PER_PAGE = 5
RANDOM_SEED = 42
OVERLAY_ALPHA = 0.45
CMAP_NAME = 'magma'
FIG_DPI = 160
SAVE_OUTPUTS = True

In [ ]:
# EDIT THESE PATHS
HEATMAPS_DIR = Path('/path/to/itc_margin_heatmaps')
LABELS_CSV = Path('/path/to/vindr_labels.csv')
IMAGES_ROOT = Path('/path/to/vindr_256_pngs')
ANNOTATIONS_CSV = Path('/path/to/vindr_annotations.csv')
IMAGE_METADATA_CSV = Path('/path/to/vindr_image_metadata.csv')
OUTPUT_DIR = Path('/path/to/itc_margin_visualization')

for path in [HEATMAPS_DIR, LABELS_CSV, IMAGES_ROOT, ANNOTATIONS_CSV, IMAGE_METADATA_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Heatmaps:', HEATMAPS_DIR)
print('Outputs: ', OUTPUT_DIR)

## Load labels, annotations, and original image dimensions

In [ ]:
def choose_column(df, candidates, purpose):
    lookup = {str(c).casefold(): c for c in df.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    raise KeyError(f'Cannot find {purpose}. Available columns: {list(df.columns)}')

labels_df = pd.read_csv(LABELS_CSV)
label_id_col = labels_df.columns[0]
labels_df = labels_df.rename(columns={label_id_col: 'image_id'})
labels_df['image_id'] = labels_df['image_id'].astype(str)
for label in TARGET_LABELS:
    if label not in labels_df.columns:
        raise KeyError(f'{label!r} missing from labels CSV')
    labels_df[label] = pd.to_numeric(labels_df[label], errors='raise')
    if not labels_df[label].isin([0, 1]).all():
        raise ValueError(f'{label} is not binary')

ann_raw = pd.read_csv(ANNOTATIONS_CSV)
ann_map = {
    'image_id': choose_column(ann_raw, ['image_id', 'imageid'], 'annotation image ID'),
    'class_name': choose_column(ann_raw, ['class_name', 'class', 'label'], 'annotation class'),
    'x_min': choose_column(ann_raw, ['x_min', 'xmin', 'x1'], 'x_min'),
    'y_min': choose_column(ann_raw, ['y_min', 'ymin', 'y1'], 'y_min'),
    'x_max': choose_column(ann_raw, ['x_max', 'xmax', 'x2'], 'x_max'),
    'y_max': choose_column(ann_raw, ['y_max', 'ymax', 'y2'], 'y_max'),
}
annotations_df = ann_raw[list(ann_map.values())].rename(columns={v: k for k, v in ann_map.items()})
annotations_df['image_id'] = annotations_df['image_id'].astype(str)

meta_raw = pd.read_csv(IMAGE_METADATA_CSV)
meta_map = {
    'image_id': choose_column(meta_raw, ['image_id', 'imageid'], 'metadata image ID'),
    'width': choose_column(meta_raw, ['width', 'image_width', 'original_width', 'w'], 'original width'),
    'height': choose_column(meta_raw, ['height', 'image_height', 'original_height', 'h'], 'original height'),
}
metadata_df = meta_raw[list(meta_map.values())].rename(columns={v: k for k, v in meta_map.items()})
metadata_df['image_id'] = metadata_df['image_id'].astype(str)
if metadata_df['image_id'].duplicated().any():
    raise ValueError('Image metadata contains duplicate image IDs')
metadata_lookup = metadata_df.set_index('image_id')[['width', 'height']].to_dict('index')
print(f'Labels={len(labels_df):,}; annotations={len(annotations_df):,}; metadata={len(metadata_df):,}')

## Match label rows to saved heatmaps

In [ ]:
labels_df = labels_df.copy()
labels_df['image_path'] = labels_df['image_id'].map(lambda x: str(IMAGES_ROOT / f'{x}.png'))
labels_df['heatmap_path'] = labels_df['image_id'].map(lambda x: str(HEATMAPS_DIR / f'{x}.pt'))
labels_df['image_exists'] = labels_df['image_path'].map(lambda x: Path(x).is_file())
labels_df['heatmap_exists'] = labels_df['heatmap_path'].map(lambda x: Path(x).is_file())
display(labels_df[['image_exists', 'heatmap_exists']].value_counts().rename('count').to_frame())
usable_df = labels_df[labels_df['image_exists'] & labels_df['heatmap_exists']].copy().reset_index(drop=True)
if usable_df.empty:
    raise RuntimeError('No image has both a PNG and a margin heatmap file')
print(f'Usable images: {len(usable_df):,}')

## Strictly validate every `.pt` file and the margin classifier math

In [ ]:
def safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def as_numpy(value):
    if torch.is_tensor(value):
        value = value.detach().cpu().float().numpy()
    return np.asarray(value, dtype=np.float32).squeeze()

def validate_file(path, expected_image_id, csv_row):
    item = safe_torch_load(Path(path))
    metadata = item.get('__metadata__')
    if not isinstance(metadata, dict):
        raise ValueError('Missing __metadata__ dictionary')
    checks = {
        'image_id': str(metadata.get('image_id')) == str(expected_image_id),
        'method': metadata.get('method') == EXPECTED_METHOD,
        'version': metadata.get('implementation_version') == EXPECTED_VERSION,
        'final_layer': int(metadata.get('vit_layer_index', -2)) == int(metadata.get('num_vit_layers', -1)) - 1,
        'target': metadata.get('target') == '(positive_similarity-negative_similarity)/temperature',
        'no_itm': metadata.get('uses_itm') is False,
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise ValueError(f'Metadata checks failed: {failed}; metadata={metadata}')
    temperature = float(metadata['temperature'])
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError(f'Invalid temperature {temperature}')
    findings = {}
    for label in TARGET_LABELS:
        if label not in item:
            raise KeyError(f'Missing finding {label!r}')
        finding = item[label]
        required = ['ground_truth', 'positive_prompt', 'negative_prompt', 'positive_similarity',
                    'negative_similarity', 'margin', 'classification_logit', 'positive_probability',
                    'cam_signed_raw', 'cam_positive_raw', 'cam_vis', 'cam_vis_up']
        missing = [key for key in required if key not in finding]
        if missing:
            raise KeyError(f'{label}: missing {missing}')
        gt = float(finding['ground_truth'])
        if gt != float(csv_row[label]):
            raise ValueError(f'{label}: PT GT={gt} != CSV GT={csv_row[label]}')
        positive_similarity = float(finding['positive_similarity'])
        negative_similarity = float(finding['negative_similarity'])
        margin = float(finding['margin'])
        logit = float(finding['classification_logit'])
        probability = float(finding['positive_probability'])
        expected_margin = positive_similarity - negative_similarity
        expected_logit = expected_margin / temperature
        expected_probability = 1.0 / (1.0 + np.exp(-expected_logit))
        if not np.isclose(margin, expected_margin, rtol=1e-5, atol=1e-7):
            raise ValueError(f'{label}: inconsistent margin')
        if not np.isclose(logit, expected_logit, rtol=1e-5, atol=1e-6):
            raise ValueError(f'{label}: inconsistent classification logit')
        if not np.isclose(probability, expected_probability, rtol=1e-5, atol=1e-6):
            raise ValueError(f'{label}: inconsistent positive probability')
        arrays = {key: as_numpy(finding[key]) for key in ['cam_signed_raw', 'cam_positive_raw', 'cam_vis', 'cam_vis_up']}
        for key in ['cam_signed_raw', 'cam_positive_raw', 'cam_vis']:
            if arrays[key].shape != (16, 16):
                raise ValueError(f'{label}: {key} has shape {arrays[key].shape}')
        if arrays['cam_vis_up'].shape != (EXPECTED_IMAGE_RES, EXPECTED_IMAGE_RES):
            raise ValueError(f"{label}: cam_vis_up has shape {arrays['cam_vis_up'].shape}")
        if not np.isfinite(np.concatenate([a.ravel() for a in arrays.values()])).all():
            raise ValueError(f'{label}: CAM contains NaN or infinity')
        if arrays['cam_positive_raw'].min() < -1e-12 or arrays['cam_vis'].min() < -1e-6 or arrays['cam_vis'].max() > 1 + 1e-6:
            raise ValueError(f'{label}: invalid positive/normalized CAM range')
        findings[label] = {**finding, **arrays}
    return metadata, findings

validation_records, validation_errors = [], []
for position, (_, row) in enumerate(usable_df.iterrows(), start=1):
    try:
        metadata, findings = validate_file(row['heatmap_path'], row['image_id'], row)
        record = {'image_id': row['image_id'], 'heatmap_path': row['heatmap_path'],
                  'view_type': metadata['view_type'], 'temperature': float(metadata['temperature'])}
        for label in TARGET_LABELS:
            f = findings[label]; slug = label.lower().replace(' ', '_')
            positive_raw = f['cam_positive_raw']; signed_raw = f['cam_signed_raw']
            record.update({
                f'{slug}_gt': int(f['ground_truth']),
                f'{slug}_positive_similarity': float(f['positive_similarity']),
                f'{slug}_negative_similarity': float(f['negative_similarity']),
                f'{slug}_margin': float(f['margin']),
                f'{slug}_logit': float(f['classification_logit']),
                f'{slug}_probability': float(f['positive_probability']),
                f'{slug}_raw_max': float(positive_raw.max()),
                f'{slug}_raw_mean': float(positive_raw.mean()),
                f'{slug}_raw_sum': float(positive_raw.sum()),
                f'{slug}_signed_positive_fraction': float(np.mean(signed_raw > 0)),
            })
        validation_records.append(record)
    except Exception as error:
        validation_errors.append({'image_id': row['image_id'], 'heatmap_path': row['heatmap_path'], 'error': repr(error)})
    if position % 250 == 0 or position == len(usable_df):
        print(f'Checked {position:,}/{len(usable_df):,} | errors={len(validation_errors):,}')
validation_df = pd.DataFrame(validation_records)
validation_errors_df = pd.DataFrame(validation_errors)
print(f'Valid={len(validation_df):,}; invalid={len(validation_errors_df):,}')
if not validation_errors_df.empty:
    display(validation_errors_df.head(20))
    raise RuntimeError('At least one heatmap failed validation')
print('All files passed structural and classifier-math validation.')

## Score and attribution diagnostics by ground truth

In [ ]:
for label in TARGET_LABELS:
    slug = label.lower().replace(' ', '_')
    columns = [f'{slug}_positive_similarity', f'{slug}_negative_similarity', f'{slug}_margin',
               f'{slug}_probability', f'{slug}_raw_max', f'{slug}_raw_mean',
               f'{slug}_signed_positive_fraction']
    print(f'\n{label}')
    display(validation_df.groupby(f'{slug}_gt')[columns].agg(['count', 'median', 'mean', 'min', 'max']))

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=FIG_DPI)
for row_index, label in enumerate(TARGET_LABELS):
    slug = label.lower().replace(' ', '_')
    for gt, color, name in [(0, 'steelblue', 'GT absent'), (1, 'darkorange', 'GT present')]:
        subset = validation_df[validation_df[f'{slug}_gt'] == gt]
        axes[row_index, 0].hist(subset[f'{slug}_margin'], bins=40, alpha=.55, color=color, label=name)
        axes[row_index, 1].hist(subset[f'{slug}_probability'], bins=40, alpha=.55, color=color, label=name)
        axes[row_index, 2].hist(np.log10(subset[f'{slug}_raw_max'] + 1e-12), bins=40, alpha=.55, color=color, label=name)
    axes[row_index, 0].set_title(f'{label}: positive − negative similarity'); axes[row_index, 0].set_xlabel('Raw margin')
    axes[row_index, 1].set_title(f'{label}: classifier score'); axes[row_index, 1].set_xlabel('Positive probability')
    axes[row_index, 2].set_title(f'{label}: positive CAM magnitude'); axes[row_index, 2].set_xlabel('log10(raw max + 1e-12)')
    for ax in axes[row_index]: ax.legend(); ax.set_ylabel('Images')
plt.tight_layout()
if SAVE_OUTPUTS: fig.savefig(OUTPUT_DIR / 'margin_cam_strength_diagnostics.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## Create four GT strata and select 50 reproducible shared cases

In [ ]:
def assign_stratum(row):
    pair = (int(row[TARGET_LABELS[0]]), int(row[TARGET_LABELS[1]]))
    return {(0,0): 'Neither', (1,0): 'Cardiomegaly only', (0,1): 'Pleural effusion only', (1,1): 'Both'}[pair]

usable_df['stratum'] = usable_df.apply(assign_stratum, axis=1)
strata = ['Neither', 'Cardiomegaly only', 'Pleural effusion only', 'Both']
display(usable_df['stratum'].value_counts().reindex(strata, fill_value=0).rename('available').to_frame())

def stratified_sample(df, n, seed):
    n = min(n, len(df)); base, remainder = divmod(n, len(strata))
    selected_parts, selected_ids = [], set()
    for index, stratum in enumerate(strata):
        candidates = df[df['stratum'] == stratum]
        take = min(base + (index < remainder), len(candidates))
        if take:
            part = candidates.sample(take, random_state=seed + index)
            selected_parts.append(part); selected_ids.update(part['image_id'])
    selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else df.iloc[:0].copy()
    if len(selected) < n:
        filler = df[~df['image_id'].isin(selected_ids)].sample(n - len(selected), random_state=seed + 100)
        selected = pd.concat([selected, filler], ignore_index=True)
    order = {name: i for i, name in enumerate(strata)}
    return selected.assign(_order=selected['stratum'].map(order)).sort_values(['_order', 'image_id']).drop(columns='_order').reset_index(drop=True)

selected_df = stratified_sample(usable_df, NUM_IMAGES, RANDOM_SEED)
display(selected_df['stratum'].value_counts().reindex(strata, fill_value=0).rename('selected').to_frame())
display(selected_df[TARGET_LABELS].agg(['sum', 'count']).T.assign(negative=lambda x: x['count']-x['sum']).rename(columns={'sum':'positive'})[['positive','negative']])

## Image, overlay, and GT-box helpers

In [ ]:
def load_image(image_id, metadata=None):
    with Image.open(IMAGES_ROOT / f'{image_id}.png') as handle:
        image = handle.convert('RGB')
    # Reproduce the actual model input for lung/heart extractions.
    if metadata is not None and metadata.get('view_type', 'original') != 'original':
        mask_path = metadata.get('mask_path')
        if not mask_path or not Path(mask_path).is_file():
            raise FileNotFoundError(f'Mask from PT metadata is unavailable: {mask_path}')
        with Image.open(mask_path) as handle:
            mask = handle.convert('L')
        if mask.size != image.size:
            raise ValueError(f'Image/mask size mismatch for {image_id}: {image.size} vs {mask.size}')
        image = Image.composite(image, Image.new('RGB', image.size), mask)
    return image

def resize_cam(cam, size_wh):
    cam = np.clip(np.nan_to_num(as_numpy(cam), nan=0., posinf=1., neginf=0.), 0., 1.)
    if cam.shape == (size_wh[1], size_wh[0]): return cam
    return np.asarray(Image.fromarray(np.round(cam*255).astype(np.uint8), mode='L').resize(size_wh, Image.Resampling.BILINEAR), dtype=np.float32)/255.

def make_overlay(image, cam, alpha=OVERLAY_ALPHA):
    rgb = np.asarray(image, dtype=np.float32)/255.; cam = resize_cam(cam, image.size)
    color = colormaps[CMAP_NAME](cam)[..., :3]; alpha_map = alpha * cam[..., None]
    return np.clip((1-alpha_map)*rgb + alpha_map*color, 0, 1)

def get_boxes(image_id, label, displayed_size):
    subset = annotations_df[(annotations_df.image_id == str(image_id)) & (annotations_df.class_name == label)]
    if subset.empty: return []
    if str(image_id) not in metadata_lookup: raise KeyError(f'Missing original dimensions for {image_id}')
    original = metadata_lookup[str(image_id)]; sx = displayed_size[0]/float(original['width']); sy = displayed_size[1]/float(original['height'])
    boxes = []
    for _, row in subset.iterrows():
        x1, y1 = np.clip(float(row.x_min)*sx, 0, displayed_size[0]), np.clip(float(row.y_min)*sy, 0, displayed_size[1])
        x2, y2 = np.clip(float(row.x_max)*sx, 0, displayed_size[0]), np.clip(float(row.y_max)*sy, 0, displayed_size[1])
        if x2 > x1 and y2 > y1: boxes.append((x1,y1,x2,y2))
    return boxes

def draw_boxes(ax, boxes, color):
    for x1,y1,x2,y2 in boxes:
        ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2))

## Inspect one case in full detail (signed and positive margin attribution)

In [ ]:
CASE_INDEX = 0  # 0 to len(selected_df)-1
row = selected_df.iloc[CASE_INDEX]; image_id = row['image_id']
metadata, findings = validate_file(row['heatmap_path'], image_id, row); image = load_image(image_id, metadata)
fig, axes = plt.subplots(2, 5, figsize=(17, 7), dpi=FIG_DPI)
for i, label in enumerate(TARGET_LABELS):
    f = findings[label]; boxes = get_boxes(image_id, label, image.size)
    signed = f['cam_signed_raw']; signed_limit = max(abs(float(signed.min())), abs(float(signed.max())), 1e-12)
    axes[i,0].imshow(image); draw_boxes(axes[i,0], boxes, 'lime'); axes[i,0].set_title(f'Input + {label} GT')
    axes[i,1].imshow(signed, cmap='coolwarm', vmin=-signed_limit, vmax=signed_limit, interpolation='nearest'); axes[i,1].set_title('Signed 16×16\nred=favors present, blue=favors absent')
    axes[i,2].imshow(f['cam_vis'], cmap=CMAP_NAME, vmin=0, vmax=1, interpolation='nearest'); axes[i,2].set_title('ReLU-positive 16×16')
    axes[i,3].imshow(f['cam_vis_up'], cmap=CMAP_NAME, vmin=0, vmax=1); axes[i,3].set_title(f"Upsampled | GT={int(f['ground_truth'])}")
    axes[i,4].imshow(make_overlay(image, f['cam_vis_up'])); draw_boxes(axes[i,4], boxes, 'lime')
    axes[i,4].set_title(f"Overlay\nmargin={f['margin']:.4f} | p+={f['positive_probability']:.4f}\nraw max={f['cam_positive_raw'].max():.2e}")
    for ax in axes[i]: ax.axis('off')
fig.suptitle(f'{CASE_INDEX+1:02d}. {image_id} | {row.stratum} | T={metadata["temperature"]:.5f}', fontweight='bold')
plt.tight_layout()
if SAVE_OUTPUTS: fig.savefig(OUTPUT_DIR / f'single_case_{CASE_INDEX+1:02d}_{image_id}.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## Create the 50-image combined gallery

In [ ]:
def create_gallery_page(page_df, page_number):
    fig, axes = plt.subplots(len(page_df), 5, figsize=(18, 3.5*len(page_df)), dpi=FIG_DPI, squeeze=False)
    for r, (_, row) in enumerate(page_df.iterrows()):
        image_id = row.image_id; metadata, f = validate_file(row.heatmap_path, image_id, row); image = load_image(image_id, metadata)
        cardio, effusion = f[TARGET_LABELS[0]], f[TARGET_LABELS[1]]
        cardio_boxes = get_boxes(image_id, TARGET_LABELS[0], image.size); effusion_boxes = get_boxes(image_id, TARGET_LABELS[1], image.size)
        axes[r,0].imshow(image); draw_boxes(axes[r,0], cardio_boxes, 'lime'); draw_boxes(axes[r,0], effusion_boxes, 'cyan')
        axes[r,0].set_title(f'{image_id}\n{row.stratum}\nlime=Cardio | cyan=Effusion', fontsize=8)
        axes[r,1].imshow(cardio['cam_vis'], cmap=CMAP_NAME, vmin=0, vmax=1, interpolation='nearest')
        axes[r,1].set_title(f"Cardiomegaly 16×16 | GT={int(cardio['ground_truth'])}\nmargin={cardio['margin']:.4f} | p+={cardio['positive_probability']:.4f}\nraw max={cardio['cam_positive_raw'].max():.2e}", fontsize=8)
        axes[r,2].imshow(make_overlay(image, cardio['cam_vis_up'])); draw_boxes(axes[r,2], cardio_boxes, 'lime'); axes[r,2].set_title('Cardiomegaly overlay', fontsize=8)
        axes[r,3].imshow(effusion['cam_vis'], cmap=CMAP_NAME, vmin=0, vmax=1, interpolation='nearest')
        axes[r,3].set_title(f"Pleural effusion 16×16 | GT={int(effusion['ground_truth'])}\nmargin={effusion['margin']:.4f} | p+={effusion['positive_probability']:.4f}\nraw max={effusion['cam_positive_raw'].max():.2e}", fontsize=8)
        axes[r,4].imshow(make_overlay(image, effusion['cam_vis_up'])); draw_boxes(axes[r,4], effusion_boxes, 'cyan'); axes[r,4].set_title('Pleural-effusion overlay', fontsize=8)
        for ax in axes[r]: ax.axis('off')
    fig.suptitle(f'ALBEF ITC contrastive-margin Grad-CAM — page {page_number:02d}', fontsize=15, fontweight='bold', y=1.002)
    plt.tight_layout(); return fig

for page_index in range(math.ceil(len(selected_df)/CASES_PER_PAGE)):
    page = selected_df.iloc[page_index*CASES_PER_PAGE:(page_index+1)*CASES_PER_PAGE]
    fig = create_gallery_page(page, page_index+1)
    if SAVE_OUTPUTS:
        path = OUTPUT_DIR / f'margin_gallery_page_{page_index+1:02d}.png'; fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight'); print('Saved:', path)
    plt.show(); plt.close(fig)

## Save selected cases and all numerical diagnostics

In [ ]:
diagnostic_columns = ['image_id'] + [c for c in validation_df.columns if c != 'image_id']
selection_output = selected_df[['image_id', *TARGET_LABELS, 'stratum', 'image_path', 'heatmap_path']].merge(validation_df, on=['image_id', 'heatmap_path'], how='left', validate='one_to_one')
selected_path = OUTPUT_DIR / 'selected_50_margin_cases.csv'
validation_path = OUTPUT_DIR / 'all_margin_heatmap_validation_diagnostics.csv'
selection_output.to_csv(selected_path, index=False); validation_df.to_csv(validation_path, index=False)
print('Saved:', selected_path); print('Saved:', validation_path)
display(selection_output.head(10))

## Interpretation checklist

- `margin > 0` and `p+ > 0.5` mean the model prefers the positive prompt.
- The signed map uses red for contributions favoring the positive prompt and blue for contributions favoring the negative prompt.
- The standard overlay displays only ReLU-positive attribution.
- Per-image normalization still makes each nonzero positive map visually intense; always read `raw max` with the overlay.
- Compare these exact selected cases with the earlier positive-similarity notebook before deciding whether the contrastive target improves localization.